# 03 · Synthetic mechanisms: aggregation, adaptation terms, attention and alignment

This notebook reproduces, on the 24 synthetic units of the shared pool:

* **Table `tab:synthetic-attention-variants`**: MIL-CREDA with six aggregations (relevance + consensus, relevance only, the attention and gated attention of Ilse et al., mean, sum): balanced accuracy (BA), Holm-corrected Wilcoxon test and Cliff's delta against the reference, mean rank, and the AUC of the attention weights against the artifact and class-event masks;
* **Table `tab:synthetic-term-ablation`**: MIL-CREDA with both adaptation terms, the global term only, the local term only, and no adaptation;
* **Figure `fig:synthetic-attention-signal`**: AF3 and T7 traces and the in-bag attention of MIL-CREDA, UDA-DDA and the ideal weighting for one simulated subject with ADHD;
* **Figure `fig:synthetic-alignment`**: bag representations of MIL-CREDA along phase 2 (UMAP) and the distance of two subjects to their own source pair;
* every number of the paragraphs that discuss them in *Results and Discussion*.

**What runs.** One unit is one call to the paper's own entrypoint `MIL_CREDA_EEG.pool_aggregation.run`. At the configuration that the label-free criterion cef picked for MIL-CREDA in the shared pool (with its training seed, 100 phase-2 epochs and annealed schedule), it trains the six aggregations and the two single-term variants from the unit's frozen phase-1 base; "no adaptation" is the frozen base itself. It also records the attention weights of every subject and the bag representations of each target subject and its own source pair at the start and end of phase 2. On the figure unit `d1p5-s1p5-r0` it additionally trains UDA-DDA at its own cef pick and keeps a mid-phase-2 snapshot. This notebook only drives the entrypoint, joins its outputs and draws the tables and figures.

**Data.** Simulated: every unit is generated by `MIL_CREDA_EEG.eeg_sim` (14-channel Emotiv montage, 128 Hz, class events, blinks and muscle artifacts, a device shift between source and target), seeded, so no download is needed. The class effect $d_{\mathrm{C}}\in\{0.6, 1, 1.5, 2.5\}$, the shift $\in\{0.5, 1.5, 3\}$ and two rotations give the 24 units `d<dC>-s<shift>-r<rotation>`.

**Inputs** (tracked under `Results/paper/03_synthetic_mechanisms/inputs/`): the v82 cef picks (`milcreda_picks.json`, `comparator_picks.json`) and the 24 frozen v82 phase-1 bases (`bases/<unit>-s1.pt`) that the paper's jobs loaded. They come from commit `d75c6c2` (tag `archive/ieee-pool-slim`); stage 01 (`01_Phase1_Bases`) is how the bases themselves are trained.

In [ ]:
import os, sys
from pathlib import Path

# src/ of the repository, from a local checkout or a Kaggle clone of
# https://github.com/Daprosero/Domain_Adaptation
here = Path.cwd()
if not (here / "src").exists() and not (here.parents[1] / "src").exists() and Path("/kaggle/working").exists():
    if not Path("/kaggle/working/Domain_Adaptation").exists():
        os.system("git clone --depth 1 https://github.com/Daprosero/Domain_Adaptation /kaggle/working/Domain_Adaptation")
        os.system("pip install -q mne==1.10.1 neurodsp==2.2.1 umap-learn")   # the paper jobs' extra requirements
    here = Path("/kaggle/working/Domain_Adaptation/MIL-CREDA-EEG/Notebooks")
root = next(p for p in (here, *here.parents) if (p / "src" / "MIL_CREDA_EEG").exists())
sys.path.insert(0, str(root / "src"))

from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage03_mechanisms as stage
print("repository:", root)
print("results:", stage.stage_dir())

## Parameters

The settings are those of the paper's Kaggle jobs `am-<unit>` (`tools/kaggle/am-<unit>/run-config.json`, commit `d75c6c2`): module `MIL_CREDA_EEG.pool_aggregation`, function `run`, `procs=4` worker processes with `threads=1` torch thread each, on CPU. Everything else is the entrypoint's default: the eight variants of `pool_aggregation.ALL`, 100 phase-2 epochs (`criteria_pool.EPOCHS2`), 120 evaluation subjects (`oracle_screen.EVAL_SUBJECTS`), and the pick's own hyperparameters and training seed (`criteria_pool.train_seed(1, trial)`). The entrypoint refuses a pick whose hyperparameters differ from the pool's draw at its trial.

**Cost.** The paper ran one Kaggle CPU notebook per unit, 24 jobs spread over several accounts. The stored records add up to about 14.2 h of single-process training (24 to 47 min per unit, 36 min on average); with 4 processes a unit takes roughly 10 to 15 min plus about 1 min of simulation. Running all 24 units in one account is therefore about 4 to 6 h of CPU; `SPLIT = (k, n)` runs units k, k+n, ... so n accounts share them.

`SMOKE = True` (or `REPRO_SMOKE=1`) instead trains one unit with two variants, one epoch and six evaluation subjects (about 70 s on a laptop CPU) into a temporary folder, only to prove the wiring; it never touches the stored results.

In [ ]:
SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"

print("units:", len(stage.UNITS), stage.UNITS)
print("this run:", stage.UNITS[:1] if SMOKE else repro.portion(stage.UNITS, SPLIT))
print("entrypoint kwargs:", stage.SMOKE_KWARGS if SMOKE else stage.RUN_KWARGS)
picks = __import__("json").loads((stage.inputs_dir() / "milcreda_picks.json").read_text())
print("figure-unit pick:", picks["units"][stage.FIGURE_UNIT]["trial"], picks["units"][stage.FIGURE_UNIT]["params"])

## Run

Each unit writes `Results/paper/03_synthetic_mechanisms/raw/am-<unit>/`: `aggregation-<unit>.json` (one row per variant: BA and AUC on the 120 evaluation subjects, BA on the 12 target-test subjects, per-subject attention AUCs, own-pair distances), `attention-<unit>.npz` (every per-window weight with the artifact and event masks) and `ownpair-<unit>.npz`, plus `umap-<unit>.npz` on the figure unit. `reproduces` is true when the reference run's BA equals the pick's BA in the pool, as it did for all 24 paper runs. A rerun resumes where it stopped.

In [ ]:
done = stage.run(split=SPLIT, smoke=SMOKE)
for d in done:
    print(d)

## Merge

When several accounts ran portions, copy each account's `raw/am-<unit>/` folders into this repository's `Results/paper/03_synthetic_mechanisms/raw/` (the folder names never collide) and run `merge`. It refuses until all 24 units are present, then writes `summary/`: the 24 records (gzipped), the figure unit's sidecars, the figure subject's signal excerpt (rebuilt from the simulator and checked against the run's masks), the UMAP embedding of the alignment figure (`n_neighbors=15, min_dist=0.3, random_state=0`) and `statistics.json` (Friedman, Holm-corrected Wilcoxon, Cliff's delta, ranks, attention AUCs, own-pair distances).

The tracked `summary/` was produced by `stage.merge(raw="Results/v82-mechanisms/agg-out")` from the paper campaign's own outputs. In smoke mode this cell is skipped, so the stored summary stays the paper's.

In [ ]:
if SMOKE:
    print("smoke run: nothing to merge; the stored summary is the paper campaign's")
else:
    print(stage.merge())

## Render

Tables (LaTeX as in the paper, CSV at full precision) and figures (PDF and PNG, the paper's one-column versions) from `summary/` alone; no training.

In [ ]:
from IPython.display import Image, display

out = stage.render()
for p in out["tables"]:
    if p.endswith(".tex"):
        print(Path(p).name); print(Path(p).read_text())
for p in out["figures"]:
    if p.endswith(".png"):
        print(Path(p).name); display(Image(filename=p))

## Numbers next to the paper

Every value below is quoted from `paper/main.tex` (the tables and the paragraphs that discuss them) and is rounded here to the precision the paper prints, so the tolerance is 0. The one exception is the Holm p of the mean aggregation: the run gives 0.00749, which the paper prints as 0.008 (rounded from the log's 0.0075); its tolerance is one unit of the last digit. Runs of this notebook from scratch should give the same numbers, since every unit is seeded and the paper's 24 runs reproduced their pool picks exactly; small differences can still come from other CPUs or library versions.

In [ ]:
rows = repro.compare(stage.paper_checks())
print(sum(r["within"] for r in rows), "of", len(rows), "within tolerance")